# The Boom Was Not a Balance Sheet

1990–2007 reproducible companion. Historical observations come from checked-in government series. The final shock calculation is explicitly hypothetical.

In [ ]:
from pathlib import Path
import numpy as np
from analysis import FIGURE_NAMES, base_index, lead_lag_correlations, load_panel, price_shock_equity, percentage_equity_lost, run
from prepare_data import build_panel, serialize_panel

panel = load_panel()
assert len(panel) == 18 and int(panel[0]['year']) == 1990 and int(panel[-1]['year']) == 2007
print('Years:', len(panel), 'fields:', len(panel[0]))

## Exact source reconstruction
The annual panel can be reconstructed locally from the FRED exports and BLS transcription without online access.

In [ ]:
assert serialize_panel(build_panel()) == Path('data/annual_panel_1990_2007.csv').read_text(encoding='utf-8')
for key in ('manufacturing_employment_millions', 'real_median_household_income_2025_usd', 'household_income_gini', 'debt_to_gdp_pct'):
    print(key, panel[0][key], panel[-1][key])

## Separate normalization and non-causal diagnostics
Real GDP per person and real median household income are not the same statistical unit. Indexing does not make them a decomposition. First differencing does not identify causality.

In [ ]:
gdp_index = base_index([r['real_gdp_per_capita_2017_usd'] for r in panel])
income_index = base_index([r['real_median_household_income_2025_usd'] for r in panel])
print('2007 indices (1990=100):', round(float(gdp_index[-1]), 2), round(float(income_index[-1]), 2))
print('Exploratory lag correlations, not causal estimates:', lead_lag_correlations(panel))

## Hypothetical housing collateral
Fixed mortgage principal and no behavioral response. Negative equity is not a default probability.

In [ ]:
shock = np.array([0.10])
for initial_ltv in (0.60, 0.80, 0.95):
    equity = price_shock_equity(initial_ltv, shock)[0]
    loss = percentage_equity_lost(initial_ltv, shock)[0]
    print(f'LTV {initial_ltv:.0%}: equity/original price {equity:.1%}; fraction of initial equity lost {loss:.1f}%')

## Regenerate all figures
This regenerates seven historical charts and the separately identified hypothetical chart, plus their CSV result tables.

In [ ]:
run(Path('.'))
assert all((Path('figures') / name).exists() for name in FIGURE_NAMES)
print('Eight figures verified.')